In [1]:
import pandas as pd
import time
import os
import re
import sys
from IPython.display import clear_output
# import gspread

sys.path.append('./src')
from GSheetImporter import GSheetImporter
from pullprice import yfinance_sym_dic, get_live_price

# Load arguments

In [2]:
GENERATE_MARKDOWN = True
GENERATE_HTML = True

REPORT_PATH = '../TradingAssistWebapp/pages/'
# GSHEET_CREDS = "c:/users/pbara/Documents/Python/secrets/sheets-pandas-reader-193e91a08e8e.json"
GSHEET_CREDS = '/home/pbarahimi/.credentials/gsheets.json'

# Read the trades worksheet

- Requires google account service credentials to read the full sheet regardless of the filters applied in the browser
- Refer to [`gsheet_access_instructions.txt`](https://share.gemini.google/0dLETC0zvoAe) for step-by-step instructions on how to setup access

In [3]:
SHEET_ID = "1HJ9h7UEtUQCXNA58UkZyPsHogJWBAcB1lNWt9nOPMR4"
SHEET_NAME = 'Trades'
num_cols = ['Open Price', 'Close Price', 'Commission','Risk ($)', 'Balance at Open', 'PnL']

gsheet = GSheetImporter(sheet_id=SHEET_ID, sheet_name=SHEET_NAME, credentials_path=GSHEET_CREDS)
gsheet.get_dataframe()
gsheet.to_num(num_cols)

# Keep open trades
df = gsheet.df[gsheet.df['Is Closed']==0].copy()

df.head()

,Open Time,Open Date,Account,Symbol,Volume,Open Price,Close Price,Commission,Balance at Open,Stop Loss,...,Closed,Risk ($),Potential Profit,PnL,Is Closed,Close Time,Close Date,Entry Link,Exit Link 1,Exit Link 2
119,8/24/2026 0:00:00,8/24/2026,Paper Trading #1,COF,-34.0,221.02,219.60,0.0,59800.0,,...,No,-97.92,0.0,48.28,0,,#VALUE!,Link,,
123,9/3/2026 9:59:40,9/3/2026,Paper Trading #1,TSLA,-14.0,371.81,354.08,0.0,61600.0,411.98,...,No,-562.38,1426.18,248.22,0,,#VALUE!,"Trade 1, 1h,",,
170,9/2/2026 11:10:12,9/2/2026,Paper Trading #2,CVNA,-462.0,74.59,74.59,0.0,100331.0,76.86,...,No,-1048.74,2587.2,0.00,0,,#VALUE!,"Trade 4, 15m",,
172,9/3/2026 10:11:59,9/3/2026,Paper Trading #2,TSLA,-24.0,371.81,354.08,0.0,100608.2,411.98,...,No,-964.08,2444.88,425.52,0,,#VALUE!,"Trade 1, 1h,",,
173,9/2/2026 11:24:12,9/2/2026,Tradestation - Equity,COF,-30.0,217.14,219.60,0.0,48759.0,221.69,...,No,-136.50,299.4,-73.80,0,,#VALUE!,"Trade 5, 15m",,


# Get Point Values

In [4]:
# Specify the tab name (optional, defaults to the first sheet)
SHEET_NAME = 'Symbols'

gsheet = GSheetImporter(SHEET_ID, SHEET_NAME, GSHEET_CREDS)
all_values = gsheet.get_all_values()
point_val_df = pd.DataFrame(all_values, columns=['Symbol', 'Point Value'])
point_val_df['Point Value'] = point_val_df['Point Value'].astype(float)

point_val_df.head()

,Symbol,Point Value
0,ADA,1.0
1,BTC,1.0
2,COF,1.0
3,CVNA,1.0
4,ETH,1.0


# Get Prices

In [5]:
price_df = pd.DataFrame(df['Symbol']).drop_duplicates()
price_df['Current Price'] = price_df.Symbol.apply(lambda x : get_live_price(x, yfinance_sym_dic))
price_df

,Symbol,Current Price
119,COF,219.600006
123,TSLA,354.079987
170,CVNA,74.589996


# Append Price to trades DF

In [6]:
df = pd.merge(df, price_df, on='Symbol', how='left')
df = pd.merge(df, point_val_df, on='Symbol', how='left')
df['Point Value'] = df['Point Value'].fillna(1)
df['PnL'] = (df['Volume'] * (df['Current Price']-df['Open Price']) * df['Point Value']).round(2)
df

,Open Time,Open Date,Account,Symbol,Volume,Open Price,Close Price,Commission,Balance at Open,Stop Loss,...,Potential Profit,PnL,Is Closed,Close Time,Close Date,Entry Link,Exit Link 1,Exit Link 2,Current Price,Point Value
0,8/24/2026 0:00:00,8/24/2026,Paper Trading #1,COF,-34.0,221.02,219.60,0.00,59800.00,,...,0.0,48.28,0,,#VALUE!,Link,,,219.600006,1.0
1,9/3/2026 9:59:40,9/3/2026,Paper Trading #1,TSLA,-14.0,371.81,354.08,0.00,61600.00,411.98,...,1426.18,248.22,0,,#VALUE!,"Trade 1, 1h,",,,354.079987,1.0
2,9/2/2026 11:10:12,9/2/2026,Paper Trading #2,CVNA,-462.0,74.59,74.59,0.00,100331.00,76.86,...,2587.2,0.00,0,,#VALUE!,"Trade 4, 15m",,,74.589996,1.0
3,9/3/2026 10:11:59,9/3/2026,Paper Trading #2,TSLA,-24.0,371.81,354.08,0.00,100608.20,411.98,...,2444.88,425.52,0,,#VALUE!,"Trade 1, 1h,",,,354.079987,1.0
4,9/2/2026 11:24:12,9/2/2026,Tradestation - Equity,COF,-30.0,217.14,219.60,0.00,48759.00,221.69,...,299.4,-73.80,0,,#VALUE!,"Trade 5, 15m",,,219.600006,1.0
5,9/3/2026 11:56:39,9/3/2026,Tradestation - Equity,COF,-34.0,209.07,219.60,0.00,48700.00,221.69,...,64.94,-358.02,0,,#VALUE!,,,,219.600006,1.0
6,8/24/2026 0:00:00,8/24/2026,Tradestation - Equity,CVNA,-140.0,72.32,74.59,-2.91,42336.39,81.13,...,2250.29,-317.80,0,,#VALUE!,Link,,,74.589996,1.0
7,9/2/2026 13:35:00,9/2/2026,Tradestation - Equity,CVNA,-25.0,74.81,74.59,0.00,48760.00,76.8,...,170.25,5.50,0,,#VALUE!,,,,74.589996,1.0
8,9/2/2026 13:35:00,9/2/2026,Tradestation - Equity,CVNA,-336.0,74.18,74.61,0.00,48760.00,74.49,...,876.54,-137.76,0,,#VALUE!,,,,74.589996,1.0
9,9/4/2026 11:30:12,9/4/2026,Tradestation - Equity,CVNA,-342.0,73.38,74.61,0.00,48760.00,73.5,...,237.48,-413.82,0,,#VALUE!,,,,74.589996,1.0


# Group by account and symbol to report

In [7]:
spacer_line = '\n\n' + 50 * '-' + '\n'
out = df.groupby('Account').agg({'PnL': sum}).to_string() + spacer_line
out += df.groupby('Symbol').agg({'Volume': sum, 'PnL': sum}).to_string() + spacer_line
out += df.groupby(['Symbol','Account']).agg({'Volume': sum, 'PnL': sum}).to_string() + spacer_line
out += df.groupby(['Account','Symbol']).agg({'Volume': sum, 'PnL': sum}).to_string() + spacer_line
print(out)

                           PnL
Account                       
Paper Trading #1        296.50
Paper Trading #2        425.52
Tradestation - Equity -1295.70

--------------------------------------------------
        Volume     PnL
Symbol                
COF      -98.0 -383.54
CVNA   -1305.0 -863.88
TSLA     -38.0  673.74

--------------------------------------------------
                              Volume     PnL
Symbol Account                              
COF    Paper Trading #1        -34.0   48.28
       Tradestation - Equity   -64.0 -431.82
CVNA   Paper Trading #2       -462.0    0.00
       Tradestation - Equity  -843.0 -863.88
TSLA   Paper Trading #1        -14.0  248.22
       Paper Trading #2        -24.0  425.52

--------------------------------------------------
                              Volume     PnL
Account               Symbol                
Paper Trading #1      COF      -34.0   48.28
                      TSLA     -14.0  248.22
Paper Trading #2      CVNA    -462

In [8]:
spacer_line = '\n\n<br>\n\n' 
out = df.groupby('Account').agg({'PnL': sum}).to_markdown() + spacer_line
out += df.groupby('Symbol').agg({'Volume': sum, 'PnL': sum}).to_markdown() + spacer_line
out += df.groupby(['Symbol','Account'], as_index=False).agg({'Volume': sum, 'PnL': sum}).to_markdown() + spacer_line
out += df.groupby(['Account','Symbol'], as_index=False).agg({'Volume': sum, 'PnL': sum}).to_markdown() + spacer_line

print(out)

| Account               |      PnL |
|:----------------------|---------:|
| Paper Trading #1      |   296.5  |
| Paper Trading #2      |   425.52 |
| Tradestation - Equity | -1295.7  |

<br>

| Symbol   |   Volume |     PnL |
|:---------|---------:|--------:|
| COF      |      -98 | -383.54 |
| CVNA     |    -1305 | -863.88 |
| TSLA     |      -38 |  673.74 |

<br>

|    | Symbol   | Account               |   Volume |     PnL |
|---:|:---------|:----------------------|---------:|--------:|
|  0 | COF      | Paper Trading #1      |      -34 |   48.28 |
|  1 | COF      | Tradestation - Equity |      -64 | -431.82 |
|  2 | CVNA     | Paper Trading #2      |     -462 |    0    |
|  3 | CVNA     | Tradestation - Equity |     -843 | -863.88 |
|  4 | TSLA     | Paper Trading #1      |      -14 |  248.22 |
|  5 | TSLA     | Paper Trading #2      |      -24 |  425.52 |

<br>

|    | Account               | Symbol   |   Volume |     PnL |
|---:|:----------------------|:---------|---------:|-----

In [9]:
if GENERATE_MARKDOWN:
    page_nm = 'acct_lvl_stats.md'
    with open(os.path.join(REPORT_PATH, page_nm), 'w') as f:  # Save to a file
        f.write(df.groupby('Account').agg({'PnL': sum}).to_markdown())
        
    page_nm = 'sym_lvl_stats.md'
    with open(os.path.join(REPORT_PATH, page_nm), 'w') as f:
        f.write(df.groupby('Symbol').agg({'Volume': sum, 'PnL': sum}).to_markdown())
    
    page_nm = 'sym_acct_lvl_stats.md'
    with open(os.path.join(REPORT_PATH, page_nm), 'w') as f:
        f.write(df.groupby(['Symbol','Account'], as_index=False).agg({'Volume': sum, 'PnL': sum}).to_markdown())
    
    page_nm = 'acct_sym_lvl_stats.md'
    with open(os.path.join(REPORT_PATH, page_nm), 'w') as f:
        f.write(df.groupby(['Account','Symbol'], as_index=False).agg({'Volume': sum, 'PnL': sum}).to_markdown())

In [10]:
if GENERATE_HTML:
    page_nm = 'acct_lvl_stats.html'
    with open(os.path.join(REPORT_PATH, page_nm), 'w') as f:  # Save to a file
        t = df.groupby('Account').agg({'PnL': sum})
        f.write(t.to_html(border=0, justify='left',  table_id='dataTable', classes='table table-striped table-hover'))
        
    page_nm = 'sym_lvl_stats.html'
    with open(os.path.join(REPORT_PATH, page_nm), 'w') as f:
        t = df.groupby('Symbol').agg({'Volume': sum, 'PnL': sum})
        f.write(t.to_html(border=0, justify='left',  table_id='dataTable', classes='table table-striped table-hover'))
    
    page_nm = 'sym_acct_lvl_stats.html'
    with open(os.path.join(REPORT_PATH, page_nm), 'w') as f:
        t = df.groupby(['Symbol','Account']).agg({'Volume': sum, 'PnL': sum})
        f.write(t.to_html(border=0, justify='left',  table_id='dataTable', classes='table table-striped table-hover'))
    
    page_nm = 'acct_sym_lvl_stats.html'
    with open(os.path.join(REPORT_PATH, page_nm), 'w') as f:
        t = df.groupby(['Account','Symbol']).agg({'Volume': sum, 'PnL': sum})
        f.write(t.to_html(border=0, justify='left',  table_id='dataTable', classes='table table-striped table-hover'))